# DJR WGS sex versus field sex

Compare WGS/Y-based sex assignments with field metadata. If WGS sex is missing, retain field sex.

In [ ]:
library(data.table)

wgs_file <- "/scratch/ztobias/BC_stickle_expanded_WGS/metadata/derived/wgs_sex_assignment_reconciliation_DJR.tsv"
dna24_file <- "/scratch/ztobias/Gacu_BC_txmics/metadata/2024_BC_DNA_extractions.csv"
meta25_file <- "/scratch/ztobias/Gacu_BC_txmics/metadata/BC Spring 2025 Collections and Molecular Metadata Sheet 1.csv"

wgs <- fread(wgs_file)
dna24 <- fread(dna24_file, fill = TRUE)
meta25 <- fread(meta25_file, fill = TRUE)

norm_sex <- function(x) {
  x <- tolower(trimws(as.character(x)))
  fifelse(grepl("^f|female", x), "female",
    fifelse(grepl("^m|male", x), "male", NA_character_))
}

wgs <- wgs[grepl("^(2024|2025)_DJR_", sample)]
wgs[, wgs_sex := norm_sex(assigned_sex_y_djr)]
wgs <- wgs[, .(sample, wgs_sex, y_missing_proportion,
  assignment_margin_djr, assignment_confident_djr)]

field24 <- dna24[grepl("^(2024|2025)_DJR_", sample), .(
  sample, field_sex = norm_sex(sex),
  field_notes = as.character(get("extr notes"))
)]

setnames(meta25, names(meta25)[1], "sample")
field25 <- meta25[grepl("^(2024|2025)_DJR_", sample), .(
  sample, field_sex = norm_sex(Sex),
  field_notes = paste(get("Field notes"), get("Eggs?"), get("stained sex check"), sep = " | ")
)]

field <- rbindlist(list(field24, field25), fill = TRUE)
field <- field[!duplicated(sample)]

sex_compare <- merge(field, wgs, by = "sample", all = TRUE)
sex_compare[, mismatch := !is.na(field_sex) & !is.na(wgs_sex) & field_sex != wgs_sex]
sex_compare[, final_sex := fifelse(!is.na(wgs_sex), wgs_sex, field_sex)]
sex_compare[, final_sex_source := fifelse(!is.na(wgs_sex), "WGS",
  fifelse(!is.na(field_sex), "field", NA_character_))]

cat("Mismatches:\n")
print(sex_compare[mismatch == TRUE])
cat("\nField-sex fallback samples:\n")
print(sex_compare[is.na(wgs_sex) & !is.na(field_sex)])

In [ ]:
out_dir <- "/scratch/ztobias/Gacu_BC_txmics/metadata/derived"
fwrite(sex_compare, file.path(out_dir, "DJR_sex_field_comparison.tsv"), sep = "\t")
fwrite(sex_compare[mismatch == TRUE], file.path(out_dir, "DJR_sex_mismatches.tsv"), sep = "\t")

sex_compare[, .(n = .N), by = .(final_sex_source, final_sex)]